In [1]:
import pandas as pd
import numpy as np

# Load datasets
orders = pd.read_csv("../data/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
customers = pd.read_csv("../data/olist_customers_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

# Convert dates
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Calculate delivery delay
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

# Merge customer geography
geo_data = orders[
    [
        "order_id",
        "customer_id",
        "delivery_delay_days"
    ]
].merge(
    customers[
        [
            "customer_id",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

# Add order value and freight
order_value = (
    order_items
    .groupby("order_id")
    .agg(
        revenue=("price", "sum"),
        freight=("freight_value", "sum")
    )
    .reset_index()
)

geo_data = geo_data.merge(
    order_value,
    on="order_id",
    how="left"
)

# Add review score
geo_data = geo_data.merge(
    reviews[
        ["order_id", "review_score"]
    ],
    on="order_id",
    how="left"
)

# Freight ratio
geo_data["freight_ratio"] = np.where(
    geo_data["revenue"] > 0,
    geo_data["freight"] / geo_data["revenue"] * 100,
    np.nan
)

print("Geographic dataset shape:", geo_data.shape)
print("\nColumns:")
print(geo_data.columns.tolist())

print("\nSample:")
print(geo_data.head())

Geographic dataset shape: (99992, 9)

Columns:
['order_id', 'customer_id', 'delivery_delay_days', 'customer_city', 'customer_state', 'revenue', 'freight', 'review_score', 'freight_ratio']

Sample:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

   delivery_delay_days            customer_city customer_state  revenue  \
0            -7.107488                sao paulo             SP    29.99   
1            -5.355729                barreiras             BA   118.70   
2           -17.245498               vianopolis             GO   159.90   
3           -12.980069  sao goncalo do amarante             RN  

In [2]:
state_analysis = (
    geo_data
    .groupby("customer_state")
    .agg(
        orders=("order_id", "nunique"),
        revenue=("revenue", "sum"),
        average_delivery_delay=("delivery_delay_days", "mean"),
        average_review_score=("review_score", "mean"),
        freight_ratio=("freight_ratio", "mean")
    )
    .reset_index()
)

state_analysis = state_analysis.round({
    "revenue": 2,
    "average_delivery_delay": 2,
    "average_review_score": 2,
    "freight_ratio": 2
})

print("Number of states:", len(state_analysis))

print("\nState-level customer experience:")
print(
    state_analysis
    .sort_values("orders", ascending=False)
    .to_string(index=False)
)

Number of states: 27

State-level customer experience:
customer_state  orders    revenue  average_delivery_delay  average_review_score  freight_ratio
            SP   41746 5228869.17                  -10.39                  4.17          25.16
            RJ   12852 1831678.85                  -11.05                  3.87          31.51
            MG   11635 1591518.47                  -12.55                  4.14          31.55
            RS    5466  754250.33                  -13.20                  4.13          34.03
            PR    5045  685911.51                  -12.61                  4.18          33.39
            SC    3637  522120.11                  -10.79                  4.07          32.94
            BA    3380  513182.78                  -10.09                  3.86          38.42
            DF    2140  304658.17                  -11.31                  4.06          33.43
            ES    2033  275910.68                   -9.80                  4.04          3

In [3]:
# Use the median as a balanced benchmark
rating_median = state_analysis["average_review_score"].median()
delay_median = state_analysis["average_delivery_delay"].median()
freight_median = state_analysis["freight_ratio"].median()

print("Median review score:", round(rating_median, 2))
print("Median delivery delay:", round(delay_median, 2))
print("Median freight ratio:", round(freight_median, 2))

# Flag states with worse-than-median customer experience
state_analysis["low_rating_flag"] = (
    state_analysis["average_review_score"] < rating_median
)

state_analysis["worse_delivery_flag"] = (
    state_analysis["average_delivery_delay"] > delay_median
)

state_analysis["high_freight_flag"] = (
    state_analysis["freight_ratio"] > freight_median
)

# Hotspot score
state_analysis["hotspot_score"] = (
    state_analysis["low_rating_flag"].astype(int)
    + state_analysis["worse_delivery_flag"].astype(int)
    + state_analysis["high_freight_flag"].astype(int)
)

hotspots = (
    state_analysis[
        state_analysis["hotspot_score"] >= 2
    ]
    .sort_values(
        ["hotspot_score", "orders"],
        ascending=[False, False]
    )
)

print("\nGEOGRAPHIC HOTSPOTS")
print("-------------------")

print(
    hotspots[
        [
            "customer_state",
            "orders",
            "revenue",
            "average_delivery_delay",
            "average_review_score",
            "freight_ratio",
            "hotspot_score"
        ]
    ].to_string(index=False)
)

Median review score: 4.05
Median delivery delay: -11.5
Median freight ratio: 42.77

GEOGRAPHIC HOTSPOTS
-------------------
customer_state  orders    revenue  average_delivery_delay  average_review_score  freight_ratio  hotspot_score
            MA     747  119844.10                   -8.96                  3.76          53.80              3
            PI     495   86974.08                  -10.62                  3.92          50.38              3
            AL     413   80518.77                   -8.04                  3.75          47.41              3
            SE     350   58920.85                   -9.33                  3.81          46.58              3
            RJ   12852 1831678.85                  -11.05                  3.87          31.51              2
            BA    3380  513182.78                  -10.09                  3.86          38.42              2
            ES    2033  275910.68                   -9.80                  4.04          34.58            

In [4]:
city_analysis = (
    geo_data
    .groupby(["customer_state", "customer_city"])
    .agg(
        orders=("order_id", "nunique"),
        revenue=("revenue", "sum"),
        average_delivery_delay=("delivery_delay_days", "mean"),
        average_review_score=("review_score", "mean"),
        freight_ratio=("freight_ratio", "mean")
    )
    .reset_index()
)

city_analysis = city_analysis.round({
    "revenue": 2,
    "average_delivery_delay": 2,
    "average_review_score": 2,
    "freight_ratio": 2
})

print("Number of cities:", len(city_analysis))

print("\nTop 20 cities by orders:")
print(
    city_analysis
    .sort_values("orders", ascending=False)
    .head(20)
    .to_string(index=False)
)


Number of cities: 4310

Top 20 cities by orders:
customer_state         customer_city  orders    revenue  average_delivery_delay  average_review_score  freight_ratio
            SP             sao paulo   15540 1925985.54                  -10.03                  4.16          24.43
            RJ        rio de janeiro    6882  997535.38                  -12.50                  3.90          31.11
            MG        belo horizonte    2773  357398.65                  -11.96                  4.11          30.97
            DF              brasilia    2131  303974.48                  -11.31                  4.06          33.35
            PR              curitiba    1521  212276.64                  -12.72                  4.20          31.77
            SP              campinas    1444  189522.28                   -8.83                  4.09          24.29
            RS          porto alegre    1379  191793.92                  -11.05                  4.01          32.81
            BA 

In [5]:
# Keep cities with enough orders for a meaningful comparison
city_filtered = city_analysis[
    city_analysis["orders"] >= 100
].copy()

print("Cities with at least 100 orders:", len(city_filtered))

# Calculate medians
city_rating_median = city_filtered["average_review_score"].median()
city_delay_median = city_filtered["average_delivery_delay"].median()
city_freight_median = city_filtered["freight_ratio"].median()

print("\nMedian review score:", round(city_rating_median, 2))
print("Median delivery delay:", round(city_delay_median, 2))
print("Median freight ratio:", round(city_freight_median, 2))

# Identify problem indicators
city_filtered["low_rating_flag"] = (
    city_filtered["average_review_score"] < city_rating_median
)

city_filtered["worse_delivery_flag"] = (
    city_filtered["average_delivery_delay"] > city_delay_median
)

city_filtered["high_freight_flag"] = (
    city_filtered["freight_ratio"] > city_freight_median
)

# Hotspot score
city_filtered["hotspot_score"] = (
    city_filtered["low_rating_flag"].astype(int)
    + city_filtered["worse_delivery_flag"].astype(int)
    + city_filtered["high_freight_flag"].astype(int)
)

city_hotspots = (
    city_filtered[
        city_filtered["hotspot_score"] >= 2
    ]
    .sort_values(
        ["hotspot_score", "orders"],
        ascending=[False, False]
    )
)

print("\nCITY-LEVEL HOTSPOTS")
print("-------------------")

print(
    city_hotspots[
        [
            "customer_state",
            "customer_city",
            "orders",
            "revenue",
            "average_delivery_delay",
            "average_review_score",
            "freight_ratio",
            "hotspot_score"
        ]
    ].to_string(index=False)
)

Cities with at least 100 orders: 141

Median review score: 4.09
Median delivery delay: -10.64
Median freight ratio: 29.75

CITY-LEVEL HOTSPOTS
-------------------
customer_state           customer_city  orders   revenue  average_delivery_delay  average_review_score  freight_ratio  hotspot_score
            BA                salvador    1245 181874.19                   -8.75                  3.72          38.06              3
            RJ                 niteroi     849 118386.90                   -9.80                  3.95          31.40              3
            GO                 goiania     692 107042.91                  -10.30                  3.97          31.75              3
            CE               fortaleza     654  97868.06                   -9.00                  3.81          40.30              3
            SC           florianopolis     570  86453.88                   -9.11                  4.00          31.74              3
            RJ             nova iguacu 

In [6]:
print("PHASE 13 — GEOGRAPHIC ANALYSIS SUMMARY")
print("=" * 50)

# Top states by revenue
print("\nTop 5 States by Revenue:")
print(
    state_analysis
    .sort_values("revenue", ascending=False)
    [["customer_state", "orders", "revenue", "average_review_score"]]
    .head(5)
    .to_string(index=False)
)

# Lowest-rated states with at least 100 orders
print("\nLowest-rated States (100+ orders):")
print(
    state_analysis[state_analysis["orders"] >= 100]
    .sort_values("average_review_score")
    [["customer_state", "orders", "revenue",
      "average_review_score", "average_delivery_delay",
      "freight_ratio"]]
    .head(10)
    .to_string(index=False)
)

# Lowest-rated cities with at least 100 orders
print("\nLowest-rated Cities (100+ orders):")
print(
    city_filtered
    .sort_values("average_review_score")
    [["customer_state", "customer_city", "orders", "revenue",
      "average_review_score", "average_delivery_delay",
      "freight_ratio"]]
    .head(10)
    .to_string(index=False)
)

# Highest freight states
print("\nHighest Freight-Ratio States:")
print(
    state_analysis
    .sort_values("freight_ratio", ascending=False)
    [["customer_state", "orders", "revenue",
      "average_review_score", "freight_ratio"]]
    .head(10)
    .to_string(index=False)
)

PHASE 13 — GEOGRAPHIC ANALYSIS SUMMARY

Top 5 States by Revenue:
customer_state  orders    revenue  average_review_score
            SP   41746 5228869.17                  4.17
            RJ   12852 1831678.85                  3.87
            MG   11635 1591518.47                  4.14
            RS    5466  754250.33                  4.13
            PR    5045  685911.51                  4.18

Lowest-rated States (100+ orders):
customer_state  orders    revenue  average_review_score  average_delivery_delay  freight_ratio
            AL     413   80518.77                  3.75                   -8.04          47.41
            MA     747  119844.10                  3.76                   -8.96          53.80
            SE     350   58920.85                  3.81                   -9.33          46.58
            CE    1336  227931.60                  3.85                  -10.12          42.77
            PA     975  179429.86                  3.85                  -13.37         